# Mission 15 - Researcher 1

- 학습 데이터 확인, 간단한 EDA, 모델 학습 및 저장 과정을 정리합니다.
- [사용 데이터](https://www.kaggle.com/datasets/nikhil7280/student-performance-multiple-linear-regression)

In [14]:
from pathlib import Path

import joblib
import pandas as pd
from dataclasses import dataclass
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [15]:
# Config 클래스 정의
@dataclass
class Config:
    base_dir: Path = Path.cwd()
    data_path: Path = base_dir / "data"
    model_path: Path = base_dir / "models"
    random_state: int = 42

    def __post_init__(self):
        self.data_path.mkdir(exist_ok=True)
        self.model_path.mkdir(exist_ok=True)

In [16]:
# 훈련 데이터 로드
cfg = Config()
train_df = pd.read_csv(cfg.data_path / 'train.csv')
train_df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,6,73,No,7,2,58.0
1,1,89,Yes,7,2,64.0
2,3,97,Yes,8,0,75.0
3,8,70,No,5,5,59.0
4,7,94,Yes,7,4,86.0


In [17]:
# 훈련 데이터 정보 확인
train_df.describe(include='all')

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
count,7000.000000,7000.000000,7000,7000.000000,7000.000000,7000.000000
unique,NaN,NaN,2,NaN,NaN,NaN
top,NaN,NaN,No,NaN,NaN,NaN
freq,NaN,NaN,3522,NaN,NaN,NaN
mean,4.950000,69.429714,NaN,6.530571,4.607429,55.095143
std,2.590621,17.289197,NaN,1.696144,2.863550,19.151574
min,1.000000,40.000000,NaN,4.000000,0.000000,10.000000
25%,3.000000,54.000000,NaN,5.000000,2.000000,40.000000
50%,5.000000,69.000000,NaN,7.000000,5.000000,55.000000
75%,7.000000,85.000000,NaN,8.000000,7.000000,70.000000


In [18]:
# 훈련 데이터 열 타입 확인
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 6 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Hours Studied                     7000 non-null   int64  
 1   Previous Scores                   7000 non-null   int64  
 2   Extracurricular Activities        7000 non-null   object 
 3   Sleep Hours                       7000 non-null   int64  
 4   Sample Question Papers Practiced  7000 non-null   int64  
 5   Performance Index                 7000 non-null   float64
dtypes: float64(1), int64(4), object(1)
memory usage: 328.2+ KB


In [19]:
# 콜롬별 분류
target_col = 'Performance Index'
categorical_features = ['Extracurricular Activities']
numeric_features = [
    'Hours Studied',
    'Previous Scores',
    'Sleep Hours',
    'Sample Question Papers Practiced',
]

In [20]:
# 훈련 데이터 라벨 분리
train_processed = train_df.drop(columns=[target_col])
label_processed = train_df[target_col]

In [21]:
# 열 전처리기
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('numeric', 'passthrough', numeric_features),
    ]
)

# 앙상블 회귀 모델 파이프라인
model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ]
)

In [22]:
# 훈련/검증 데이터 분리 및 모델 훈련
x_train, x_valid, y_train, y_valid = train_test_split(train_processed, label_processed, test_size=0.2, random_state=42)
model.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Extracurricular '
                                                   'Activities']),
                                                 ('numeric', 'passthrough',
                                                  ['Hours Studied',
                                                   'Previous Scores',
                                                   'Sleep Hours',
                                                   'Sample Question Papers '
                                                   'Practiced'])])),
                ('regressor',
                 RandomForestRegressor(n_estimators=200, n_jobs=-1,
                                       random_state=42))])

In [23]:
# 검증 및 RMSE 계산
pred = model.predict(x_valid)
rmse = root_mean_squared_error(y_valid, pred)
rmse

np.float64(2.2455744611379425)

In [24]:
# 배포용 모델 훈련 및 저장
model.fit(train_processed, label_processed)
joblib.dump(model, cfg.model_path / 'model.pkl')
cfg.model_path / 'model.pkl'

PosixPath('/home/hosung/pytorch-demo/Mission_Archive/Codeit_Mission/미션15_1팀_안호성/mission-result/researcher1/models/model.pkl')